# Evaluate Paper Submissions

Notebook này dùng metric official từ `official-evaluation-metric-text-normalization.ipynb`, sau đó chấm nhiều file submission để xuất bảng điền vào paper.

Output chính nằm trong `fillpaper/evaluate/paper_metric_outputs/`.


In [16]:
import sys
from pathlib import Path

# Nếu chạy notebook từ thư mục project root, giữ nguyên các giá trị này thường là đủ.
# Nếu chạy trên Kaggle/local ở nơi khác, sửa các path dưới đây.
def find_fillpaper_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if p.name == "fillpaper" and ((p / "test.jsonl").exists() or (p / "evaluate").exists()):
            return p
        nested = p / "fillpaper"
        if nested.exists() and ((nested / "test.jsonl").exists() or (nested / "evaluate").exists()):
            return nested
    return Path("../").resolve()

FILLPAPER_ROOT = find_fillpaper_root()
GT_JSONL = FILLPAPER_ROOT / "test.jsonl"
OUTPUT_DIR = FILLPAPER_ROOT / "evaluate" / "paper_metric_outputs"

# Thêm folder/file CSV ở đây nếu bạn lưu submission ở chỗ khác sau khi download từ Kaggle.
SUBMISSION_SEARCH_DIRS = [
    FILLPAPER_ROOT / "full_system_results",
    FILLPAPER_ROOT / "routing_ablation",
    FILLPAPER_ROOT / "qwen_ablation",
    FILLPAPER_ROOT / "module_results",
    Path("/kaggle/working"),
]
EXTRA_SUBMISSION_FILES = [
    # Path(r"C:/path/to/some_submission.csv"),
]

ROW_ID_COLUMN = "image"
IOU_THRESHOLD = 0.5
WRITE_DEBUG_DETAILS = True
ROUND_DIGITS = 4

print("FILLPAPER_ROOT:", FILLPAPER_ROOT)
print("GT_JSONL:", GT_JSONL)
print("OUTPUT_DIR:", OUTPUT_DIR)


FILLPAPER_ROOT: C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper
GT_JSONL: C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\test.jsonl
OUTPUT_DIR: C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs


In [17]:
import sys
from pathlib import Path

KAGGLE_METRIC_SOURCE = '"""\nCustom Kaggle evaluation metric for Ukrainian Handwritten Text Recognition.\n\nScore = 0.15 * Detection_F1 + 0.05 * ClassAcc + 0.30 * (1 - CER) + 0.50 * (1 - PageCER)\n\nText is normalized before CER comparison:\n  - Cyrillic/Latin lookalike characters → Cyrillic\n  - All dash types → hyphen-minus\n  - Whitespace collapse, strip\n  - Quote/apostrophe normalization\n  - Strikethrough markers ~~text~~ removed\n  - Formula: Unicode super/subscripts → ^/_ notation, single-char braces removed\n  - Tables: whitespace around pipes stripped\n\nSubmission format: CSV with columns `image` and `regions`.\nThe `regions` column contains a JSON-encoded list of region objects:\n    [{"bbox": [x1, y1, x2, y2], "type": "handwritten", "text": "..."}]\n\nRequired fields per region: bbox, type, text.\nRegion types: handwritten, printed, formula, table, annotation, image, graph.\nIf an image has no regions, use an empty list: []\n\nNote: `language` and `legibility` are GT-only attributes — participants do not need to predict them.\n\nRecent changes\n--------------\n- PageCER is now computed symmetrically: prediction regions that match a\n  non-scorable GT region (illegible / language=other / image / graph)\n  are excluded from `pred_page`, in the same way GT non-scorable regions\n  are excluded from `gt_page`. Previously, a prediction\'s text for an\n  illegible GT region inflated `pred_page` while the GT side excluded it,\n  preventing theoretically-perfect scores on pages with such regions.\n  Detection F1, classification accuracy and per-region CER are unchanged.\n\n>>> import pandas as pd\n>>> row_id_column_name = "image"\n>>> solution = pd.DataFrame({\n...     "image": ["test.jpg"],\n...     "regions": [\'[{"bbox":[50,100,850,130],"type":"handwritten","language":"uk","legibility":"legible","text":"Доброго ранку"},{"bbox":[50,150,870,180],"type":"handwritten","language":"uk","legibility":"legible","text":"Сьогодні гарна погода"},{"bbox":[50,250,650,270],"type":"printed","language":"uk","legibility":"legible","text":"Завдання 1"},{"bbox":[700,50,950,250],"type":"image","language":"uk","legibility":"legible","text":""},{"bbox":[900,950,980,990],"type":"annotation","language":"uk","legibility":"legible","text":"5"}]\'],\n... })\n>>> submission = pd.DataFrame({\n...     "image": ["test.jpg"],\n...     "regions": [\'[{"bbox":[52,101,852,131],"type":"handwritten","language":"uk","legibility":"legible","text":"Доброго ранку"},{"bbox":[50,148,860,184],"type":"handwritten","language":"uk","legibility":"legible","text":"Сьогодні гарна пагода"},{"bbox":[48,249,649,272],"type":"handwritten","language":"uk","legibility":"legible","text":"Завдання 1"},{"bbox":[710,60,945,248],"type":"image","language":"uk","legibility":"legible","text":""}]\'],\n... })\n>>> round(score(solution, submission, row_id_column_name), 4)\n0.9348\n"""\n\nimport json\nimport re\nimport pandas as pd\n\n\nclass ParticipantVisibleError(Exception):\n    pass\n\n\n# ── Text normalization ────────────────────────────────────────\n\n# ── LaTeX command → Unicode symbol mapping ────────────────────\n# Applied BEFORE Cyrillic/Latin conversion (so \\pi doesn\'t become \\рі)\n\n_LATEX_SYMBOLS = {\n    # Greek letters (common in formulas)\n    r\'\\alpha\': \'α\', r\'\\beta\': \'β\', r\'\\gamma\': \'γ\', r\'\\delta\': \'δ\',\n    r\'\\epsilon\': \'ε\', r\'\\varepsilon\': \'ε\', r\'\\zeta\': \'ζ\', r\'\\eta\': \'η\',\n    r\'\\theta\': \'θ\', r\'\\vartheta\': \'ϑ\', r\'\\iota\': \'ι\', r\'\\kappa\': \'κ\',\n    r\'\\lambda\': \'λ\', r\'\\mu\': \'μ\', r\'\\nu\': \'ν\', r\'\\xi\': \'ξ\',\n    r\'\\pi\': \'π\', r\'\\rho\': \'ρ\', r\'\\sigma\': \'σ\', r\'\\tau\': \'τ\',\n    r\'\\upsilon\': \'υ\', r\'\\phi\': \'φ\', r\'\\varphi\': \'φ\', r\'\\chi\': \'χ\',\n    r\'\\psi\': \'ψ\', r\'\\omega\': \'ω\',\n    r\'\\Gamma\': \'Γ\', r\'\\Delta\': \'Δ\', r\'\\Theta\': \'Θ\', r\'\\Lambda\': \'Λ\',\n    r\'\\Xi\': \'Ξ\', r\'\\Pi\': \'Π\', r\'\\Sigma\': \'Σ\', r\'\\Phi\': \'Φ\',\n    r\'\\Psi\': \'Ψ\', r\'\\Omega\': \'Ω\',\n    # Operators & relations\n    r\'\\cdot\': \'·\', r\'\\times\': \'×\', r\'\\div\': \'÷\', r\'\\pm\': \'±\', r\'\\mp\': \'∓\',\n    r\'\\circ\': \'∘\', r\'\\bullet\': \'•\', r\'\\star\': \'⋆\',\n    r\'\\leq\': \'≤\', r\'\\le\': \'≤\', r\'\\geq\': \'≥\', r\'\\ge\': \'≥\',\n    r\'\\neq\': \'≠\', r\'\\ne\': \'≠\', r\'\\approx\': \'≈\', r\'\\equiv\': \'≡\',\n    r\'\\sim\': \'∼\', r\'\\propto\': \'∝\',\n    # Arrows\n    r\'\\rightarrow\': \'→\', r\'\\to\': \'→\', r\'\\leftarrow\': \'←\',\n    r\'\\leftrightarrow\': \'↔\', r\'\\Rightarrow\': \'⇒\', r\'\\Leftarrow\': \'⇐\',\n    r\'\\Leftrightarrow\': \'⇔\', r\'\\implies\': \'⇒\', r\'\\iff\': \'⇔\',\n    # Set theory\n    r\'\\cap\': \'∩\', r\'\\cup\': \'∪\', r\'\\subset\': \'⊂\', r\'\\supset\': \'⊃\',\n    r\'\\subseteq\': \'⊆\', r\'\\supseteq\': \'⊇\', r\'\\in\': \'∈\', r\'\\notin\': \'∉\',\n    r\'\\emptyset\': \'∅\', r\'\\varnothing\': \'∅\',\n    r\'\\oplus\': \'⊕\', r\'\\otimes\': \'⊗\',\n    # Misc\n    r\'\\infty\': \'∞\', r\'\\partial\': \'∂\', r\'\\nabla\': \'∇\',\n    r\'\\forall\': \'∀\', r\'\\exists\': \'∃\', r\'\\neg\': \'¬\',\n    r\'\\sqrt\': \'√\', r\'\\sum\': \'∑\', r\'\\prod\': \'∏\', r\'\\int\': \'∫\',\n    r\'\\ldots\': \'…\', r\'\\dots\': \'…\', r\'\\cdots\': \'⋯\',\n    # Geometry / logic symbols\n    r\'\\therefore\': \'∴\', r\'\\because\': \'∵\',\n    r\'\\perp\': \'⊥\', r\'\\angle\': \'∠\', r\'\\parallel\': \'∥\',\n    r\'\\square\': \'□\', r\'\\Box\': \'□\', r\'\\triangle\': \'△\',\n    # Up/down arrows (often used as products of reaction)\n    r\'\\uparrow\': \'↑\', r\'\\downarrow\': \'↓\', r\'\\Uparrow\': \'⇑\', r\'\\Downarrow\': \'⇓\',\n    # Logical operators\n    r\'\\vee\': \'∨\', r\'\\wedge\': \'∧\', r\'\\lor\': \'∨\', r\'\\land\': \'∧\',\n    # Set operations\n    r\'\\setminus\': \'∖\', r\'\\backslash\': \'\\\\\',\n    # Vertical bar variants\n    r\'\\mid\': \'|\', r\'\\lvert\': \'|\', r\'\\rvert\': \'|\', r\'\\Vert\': \'‖\',\n    # Floor/ceil delimiters\n    r\'\\lfloor\': \'⌊\', r\'\\rfloor\': \'⌋\', r\'\\lceil\': \'⌈\', r\'\\rceil\': \'⌉\',\n    # Marvosym/wasysym symbols (used in genetics for sex notation)\n    r\'\\male\': \'♂\', r\'\\female\': \'♀\',\n}\n\n# LaTeX named functions: \\sin, \\cos, \\ln, \\lim, ... → strip backslash + trailing space\n# Without trailing space "\\\\sin\\\\alpha" would normalize to "sinα" while "sin α" → "sin α"\n# (mismatched). Trailing space collapses with following whitespace via _MULTI_SPACE.\n_LATEX_FUNCTION_NAMES = (\n    \'arcsin\',\'arccos\',\'arctan\',\'arcctg\',\'arccot\',\'arcsec\',\'arccsc\',\n    \'sinh\',\'cosh\',\'tanh\',\'coth\',\n    \'sin\',\'cos\',\'tan\',\'cot\',\'sec\',\'csc\',\'ctg\',\n    \'liminf\',\'limsup\',\n    \'log\',\'ln\',\'lg\',\'exp\',\'lim\',\'sup\',\'inf\',\'min\',\'max\',\'det\',\'dim\',\'gcd\',\'lcm\',\'mod\',\n    \'arg\',\'deg\',\'hom\',\'ker\',\n)\n# `(?![A-Za-z])` (not `\\b`) so that `\\lim_{x→0}` and `\\sin{x}` still match: the\n# char after the command name may be `_`/`^`/`{` which `\\b` would block.\n_LATEX_FUNCTIONS_RE = re.compile(r\'\\\\(\' + \'|\'.join(_LATEX_FUNCTION_NAMES) + r\')(?![A-Za-z])\')\n\n# \\xrightarrow{label} / \\xleftarrow{label} → arrow (label discarded)\n# Some forms have optional bracketed below-label: \\xrightarrow[below]{above}\n_XARROW_RIGHT = re.compile(r\'\\\\xrightarrow\\s*(?:\\[[^\\]]*\\])?\\s*\\{[^{}]*\\}\')\n_XARROW_LEFT = re.compile(r\'\\\\xleftarrow\\s*(?:\\[[^\\]]*\\])?\\s*\\{[^{}]*\\}\')\n\n# Sizing commands (visual hints, no semantic value) → strip\n_LATEX_SIZING = re.compile(r\'\\\\(?:big|Big|bigg|Bigg)[lr]?\\b\')\n\n# Math styles \\mathrm{}, \\mathbf{}, ..., \\operatorname{} → strip wrapper\n_MATH_STYLE = re.compile(r\'\\\\(?:mathrm|mathbf|mathit|mathbb|mathcal|mathfrak|mathsf|mathtt|operatorname|boldsymbol|pmb)\\s*\\{([^{}]*)\\}\')\n\n# Cancel/overset/underset: cancellation marks → keep base content\n# \\cancel{18} → 18 (the struck-through value is what we want to read back)\n# \\overset{a}{b} → b (b is the main symbol; a is a decoration like an oxidation state)\n# \\underset{a}{b} → b (same logic)\n_CANCEL = re.compile(r\'\\\\cancel\\s*\\{([^{}]*)\\}\')\n_OVERSET = re.compile(r\'\\\\overset\\s*\\{[^{}]*\\}\\s*\\{([^{}]*)\\}\')\n_UNDERSET = re.compile(r\'\\\\underset\\s*\\{[^{}]*\\}\\s*\\{([^{}]*)\\}\')\n# Sort by length descending so \\rightarrow matches before \\right\n_LATEX_COMMANDS_RE = re.compile(\n    \'|\'.join(re.escape(k) for k in sorted(_LATEX_SYMBOLS.keys(), key=len, reverse=True))\n)\n\n# \\text{...} → content (strip LaTeX text wrapper)\n_LATEX_TEXT_WRAPPER = re.compile(r\'\\\\text\\{([^}]*)\\}\')\n# \\left( \\right) → ( )\n_LATEX_LEFT_RIGHT = re.compile(r\'\\\\(left|right)\\s*([()|\\[\\]{}.])\')\n# LaTeX spacing commands → space or nothing\n_LATEX_SPACING = re.compile(r\'\\\\[,;:!]|\\\\quad|\\\\qquad|\\\\hspace\\{[^}]*\\}\')\n\n# Multiplication sign normalization: * and · → · (middle dot)\n_MULT_SIGNS = re.compile(r\'[*∗⋅]\')  # asterisk, combining asterisk, dot operator\n\n# Latin → Cyrillic lookalike mapping (lowercase + uppercase)\n_LATIN_TO_CYRILLIC = {\n    \'a\': \'а\', \'c\': \'с\', \'e\': \'е\', \'i\': \'і\', \'o\': \'о\',\n    \'p\': \'р\', \'x\': \'х\', \'y\': \'у\',\n    \'A\': \'А\', \'B\': \'В\', \'C\': \'С\', \'E\': \'Е\', \'H\': \'Н\',\n    \'K\': \'К\', \'M\': \'М\', \'O\': \'О\', \'P\': \'Р\', \'T\': \'Т\', \'X\': \'Х\',\n}\n\n# Unicode superscript/subscript → ASCII\n_SUPERSCRIPTS = str.maketrans(\'⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ\', \'0123456789+-=()n\')\n_SUBSCRIPTS = str.maketrans(\'₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎\', \'0123456789+-=()\')\n\n# All dash-like characters → hyphen-minus\n_DASHES = re.compile(r\'[\\u2010\\u2011\\u2012\\u2013\\u2014\\u2015\\u2212\\uFE58\\uFE63\\uFF0D]\')\n\n# Filler dashes/underscores (3+ repeated) → normalized form\n_FILLERS = re.compile(r\'[_\\-]{3,}\')\n\n# Strikethrough: ~~old~~{new} → new (correction replaces struck-through text)\n_STRIKETHROUGH_CORRECTION = re.compile(r\'~~.*?~~\\{(.*?)\\}\')\n# Strikethrough: ~~text~~ → text (standalone, no correction)\n_STRIKETHROUGH = re.compile(r\'~~(.*?)~~\')\n\n# Multiple whitespace → single space\n_MULTI_SPACE = re.compile(r\'[ \\t\\u00A0\\u2000-\\u200B\\u3000]+\')\n\n# Spaces inside brackets: "( text )" → "(text)"\n_SPACE_IN_PARENS = re.compile(r\'\\(\\s+\')\n_SPACE_IN_PARENS_R = re.compile(r\'\\s+\\)\')\n\n# Space immediately before "(" or before sub/superscript markers — strips\n# the trailing space introduced by `\\sin ` / `\\lim ` etc. so:\n#   "\\sin(x)" → "sin (x)" → "sin(x)" (matches plain "sin(x)")\n#   "\\lim_{x→0}" → "lim _{x→0}" → "lim_{x→0}"\n_SPACE_BEFORE_PAREN = re.compile(r\' +\\(\')\n_SPACE_BEFORE_SUBSUPER = re.compile(r\' +([_^])\')\n\n# LaTeX braces: x_{3} → x_3, x^{2} → x^2, S_{повн} → S_повн\n_LATEX_BRACE = re.compile(r\'([_^])\\{([^}]+)\\}\')\n\n# LaTeX table environments → PSV\n# Covers: array, tabular, matrix/pmatrix/bmatrix/vmatrix/Vmatrix/smallmatrix,\n# aligned/align/alignat/gathered/cases (multi-row alignment envs that share\n# the same \\\\ row-separator + & column-separator syntax).\n_LATEX_TABLE_ENV = re.compile(r\'\\\\begin\\{(?:array|tabular|matrix|pmatrix|bmatrix|vmatrix|Vmatrix|smallmatrix|aligned|align|alignat|gathered|cases|split)\\*?\\}(?:\\{[^}]*\\})?\\s*\')\n_LATEX_TABLE_ENV_END = re.compile(r\'\\s*\\\\end\\{(?:array|tabular|matrix|pmatrix|bmatrix|vmatrix|Vmatrix|smallmatrix|aligned|align|alignat|gathered|cases|split)\\*?\\}\')\n_LATEX_TABLE_ROW_SEP = re.compile(r\'\\s*\\\\\\\\\\s*\')\n_LATEX_TABLE_COL_SEP = re.compile(r\'\\s*&\\s*\')\n\n# Table horizontal line decorations are visual-only, no semantic value.\n# `\\hline` is a no-arg command; `\\cline{2-4}` takes a span argument.\n_TABLE_LINES = re.compile(r\'\\\\hline\\b|\\\\cline\\s*\\{[^{}]*\\}\')\n\n# `\\phantom{x}` renders invisible — no semantic value, strip.\n_PHANTOM = re.compile(r\'\\\\phantom\\s*\\{[^{}]*\\}\')\n\n# `\\underline{x}` → x  (visual underline; same treatment as `\\bar` etc.)\n_UNDERLINE = re.compile(r\'\\\\underline\\s*\\{([^{}]*)\\}\')\n\n# Student-style row separator inside plain parens: `(a b \\n c d)` means a 2D\n# matrix (one row per `\\n`). Normalize to multiline PSV like the existing\n# `(a; b)` rule does for column matrices.\n# Match an opening paren, body containing at least one literal `\\n`, closing paren.\n_PAREN_NEWLINE_ROWS = re.compile(r\'\\(([^()]*\\\\n[^()]*)\\)\')\n\n# Quotes normalization\n_QUOTES_DOUBLE = re.compile(r\'["\\u201C\\u201D\\u201E\\u00AB\\u00BB\\u2033]\')\n_QUOTES_SINGLE = re.compile(r"[\'\\u2018\\u2019\\u02BC\\u0027\\u2032]")\n\n# Pipe-separated values: strip whitespace around pipes\n_PSV_PIPE = re.compile(r\'\\s*\\|\\s*\')\n\n# \\frac{a}{b} → a/b  (applied iteratively for nested fractions)\n_FRAC = re.compile(r\'\\\\frac\\s*\\{([^{}]*)\\}\\s*\\{([^{}]*)\\}\')\n\n# \\sqrt{...} → √...  (after \\sqrt → √ symbol mapping, strip the trailing argument braces)\n_SQRT_BRACES = re.compile(r\'√\\s*\\{([^{}]*)\\}\')\n\n# Arrow decorations: \\overrightarrow{X} → →X, \\overleftarrow{X} → ←X, \\vec{X} → →X\n_ARROW_RIGHT_DECOR = re.compile(r\'\\\\(?:overrightarrow|vec)\\s*\\{([^{}]*)\\}\')\n_ARROW_LEFT_DECOR = re.compile(r\'\\\\overleftarrow\\s*\\{([^{}]*)\\}\')\n\n# Decorators: \\bar{x}, \\hat{x}, \\overline{x}, \\widetilde{x}, \\widehat{x}, \\dot{x}, \\ddot{x} → strip wrapper\n_DECORATOR = re.compile(r\'\\\\(?:bar|hat|overline|widetilde|widehat|dot|ddot)\\s*\\{([^{}]*)\\}\')\n\n# Combining diacritical marks (U+0300-036F) + symbol combining (U+20D0-20FF, includes \\vec arrow ⃗)\n# Strip after decorators to symmetrize: \\bar{x} (stripped to x) ↔ x̄ (Unicode combining macron stripped to x)\n_COMBINING_MARKS = re.compile(r\'[̀-ͯ⃐-\u20ff]\')\n\n# Plain (a; b; c) column matrix → PSV (a\\nb\\nc)  — only when paren content is purely semicolon-separated\n_PAREN_SEMI = re.compile(r\'\\(([^();]+(?:\\s*;\\s*[^();]+)+)\\)\')\n\n# Plain pipe determinant: | a b | | c d | | e f | (2+ pipe-bounded segments) → PSV\n_PIPE_MATRIX = re.compile(r\'\\|\\s*([^|\\n]+?)\\s*\\|(?:\\s*\\|\\s*[^|\\n]+?\\s*\\|)+\')\n\n\ndef _normalize_text(text: str, region_type: str = "handwritten") -> str:\n    """\n    Normalize text before CER comparison.\n    Applied identically to both GT and prediction.\n\n    >>> _normalize_text("Доброго ранку")\n    \'Доброго ранку\'\n    >>> _normalize_text("cocна")  # Latin \'c\',\'o\' → Cyrillic\n    \'сосна\'\n    >>> _normalize_text("тире — довге")  # em-dash → hyphen\n    \'тире - довге\'\n    >>> _normalize_text("~~закреслено~~ слово")\n    \'закреслено слово\'\n    >>> _normalize_text("x_{3} + y^{2}", region_type="formula")\n    \'х_3 + у^2\'\n    >>> _normalize_text("A | B | C", region_type="table")\n    \'А|В|С\'\n    >>> _normalize_text("x² + y₃", region_type="formula")\n    \'х^2 + у_3\'\n    >>> _normalize_text(\'«Привіт»\')\n    \'"Привіт"\'\n    >>> _normalize_text("( дужки )")\n    \'(дужки)\'\n    >>> _normalize_text("_____")\n    \'___\'\n    >>> _normalize_text("cat") == _normalize_text("сat")  # Latin/Cyrillic forgiven\n    True\n    >>> _normalize_text("π r^2", region_type="formula")  # Unicode π unchanged\n    \'π r^2\'\n    >>> _normalize_text("2 * 3 = 6", region_type="formula")  # * → ·\n    \'2 · 3 = 6\'\n    >>> _normalize_text("x² + y₃", region_type="formula")  # Unicode super/sub\n    \'х^2 + у_3\'\n    >>> _normalize_text("H_{2}SO_{4}", region_type="formula")  # single-char braces\n    \'Н_2SО_4\'\n    >>> _normalize_text(r"\\\\frac{1}{2}", region_type="formula")  # frac → plain\n    \'1/2\'\n    >>> _normalize_text(r"a/b", region_type="formula") == _normalize_text(r"\\\\frac{a}{b}", region_type="formula")\n    True\n    >>> _normalize_text(r"\\\\sqrt{169}", region_type="formula")\n    \'√169\'\n    >>> _normalize_text(r"\\\\bar{x}", region_type="formula")\n    \'х\'\n    >>> _normalize_text("(1; -1)", region_type="formula")  # column matrix → PSV\n    \'1\\\\n-1\'\n    >>> _normalize_text("| 3 2 | | -1 1 |", region_type="formula")  # pipe determinant → PSV\n    \'3 2\\\\n-1 1\'\n    >>> _normalize_text(r"\\\\overrightarrow{AB}", region_type="formula")  # arrow notation\n    \'→АВ\'\n    >>> _normalize_text(r"\\\\vec{a}", region_type="formula")  # vec also → arrow\n    \'→а\'\n    >>> _normalize_text(r"\\\\therefore \\\\overrightarrow{AB} \\\\perp \\\\overrightarrow{AC}", region_type="formula")\n    \'∴ →АВ ⊥ →АС\'\n    >>> _normalize_text("x̄", region_type="formula") == _normalize_text(r"\\\\bar{x}", region_type="formula")\n    True\n    """\n    if not text:\n        return ""\n\n    # 1. Strikethrough: ~~old~~{new} → new (must come before plain ~~)\n    text = _STRIKETHROUGH_CORRECTION.sub(r\'\\1\', text)\n    text = _STRIKETHROUGH.sub(r\'\\1\', text)\n\n    # 2. LaTeX normalization (BEFORE Cyrillic conversion — so \\pi doesn\'t become \\рі)\n    if region_type in ("formula", "table"):\n        # LaTeX table environments → PSV (must be before symbol conversion)\n        text = _LATEX_TABLE_ENV.sub(\'\', text)\n        text = _LATEX_TABLE_ENV_END.sub(\'\', text)\n        text = _LATEX_TABLE_ROW_SEP.sub(\'\\n\', text)\n        text = _LATEX_TABLE_COL_SEP.sub(\'|\', text)\n        # Visual-only table decorations: \\hline, \\cline{2-4}, \\phantom{x}\n        text = _TABLE_LINES.sub(\'\', text)\n        text = _PHANTOM.sub(\'\', text)\n        # \\underline{x} → x (treat like \\bar)\n        text = _UNDERLINE.sub(r\'\\1\', text)\n        # \\text{...} → content\n        text = _LATEX_TEXT_WRAPPER.sub(r\'\\1\', text)\n        # \\left( \\right) → ( )\n        text = _LATEX_LEFT_RIGHT.sub(r\'\\2\', text)\n        # Sizing hints \\big, \\Bigg, \\bigl, \\biggr, … → strip\n        text = _LATEX_SIZING.sub(\'\', text)\n        # Math styles \\mathrm{}, \\mathbf{}, ..., \\operatorname{} → strip wrapper (iterate for nested)\n        prev = None\n        while text != prev:\n            prev = text\n            text = _MATH_STYLE.sub(r\'\\1\', text)\n        # \\xrightarrow[below]{above} / \\xleftarrow → → / ←\n        text = _XARROW_RIGHT.sub(\'→\', text)\n        text = _XARROW_LEFT.sub(\'←\', text)\n        # \\cancel{x} → x; \\overset{a}{b} / \\underset{a}{b} → b\n        # Iterate because these can be nested (e.g. \\overset{\\overset{...}{|}}{X})\n        prev = None\n        while text != prev:\n            prev = text\n            text = _CANCEL.sub(r\'\\1\', text)\n            text = _OVERSET.sub(r\'\\1\', text)\n            text = _UNDERSET.sub(r\'\\1\', text)\n        # LaTeX spacing → single space\n        text = _LATEX_SPACING.sub(\' \', text)\n        # \\frac{a}{b} → a/b  (iterate for nested fractions)\n        prev = None\n        while text != prev:\n            prev = text\n            text = _FRAC.sub(r\'\\1/\\2\', text)\n        # Arrow decorations: \\overrightarrow{X}/\\vec{X} → →X, \\overleftarrow{X} → ←X\n        text = _ARROW_RIGHT_DECOR.sub(r\'→\\1\', text)\n        text = _ARROW_LEFT_DECOR.sub(r\'←\\1\', text)\n        # Decorators: \\bar{x}, \\hat{x}, \\overline{x} → strip wrapper\n        text = _DECORATOR.sub(r\'\\1\', text)\n        # Named functions: \\sin, \\cos, \\log, \\lim … → "sin ", "cos ", … (trailing space\n        # ensures "\\\\sin\\\\alpha" → "sin α" matches "sin α"; collapsed later)\n        text = _LATEX_FUNCTIONS_RE.sub(r\'\\1 \', text)\n        # LaTeX symbols → Unicode (\\pi → π, \\cdot → ·, etc.)\n        text = _LATEX_COMMANDS_RE.sub(lambda m: _LATEX_SYMBOLS[m.group()], text)\n        # \\sqrt argument braces: √{169} → √169  (after \\sqrt → √ mapping above)\n        text = _SQRT_BRACES.sub(r\'√\\1\', text)\n        # Strip Unicode combining marks (symmetric with decorator wrapper-stripping above)\n        # e.g. x̄ (x + U+0304 macron) → x; a⃗ (a + U+20D7 right arrow) → a\n        text = _COMBINING_MARKS.sub(\'\', text)\n        # Multiplication signs: * ∗ ⋅ → · (middle dot)\n        text = _MULT_SIGNS.sub(\'·\', text)\n        # Unicode superscripts → ^N\n        converted = []\n        for ch in text:\n            if ch in \'⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾ⁿ\':\n                converted.append(\'^\' + ch.translate(_SUPERSCRIPTS))\n            elif ch in \'₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎\':\n                converted.append(\'_\' + ch.translate(_SUBSCRIPTS))\n            else:\n                converted.append(ch)\n        text = \'\'.join(converted)\n        # Braces: x_{3} → x_3, S_{повн} → S_повн\n        text = _LATEX_BRACE.sub(r\'\\1\\2\', text)\n        # Plain (a; b; c) column matrix → PSV\n        text = _PAREN_SEMI.sub(lambda m: \'\\n\'.join(x.strip() for x in m.group(1).split(\';\')), text)\n        # Plain `(a b \\n c d)` 2D matrix (student notation) → multiline rows\n        text = _PAREN_NEWLINE_ROWS.sub(lambda m: \'\\n\'.join(x.strip() for x in m.group(1).split(\'\\\\n\')), text)\n        # Plain pipe determinant: | a b | | c d | | e f | → PSV (each row on its own line)\n        text = _PIPE_MATRIX.sub(lambda m: \'\\n\'.join(re.findall(r\'\\|\\s*([^|\\n]+?)\\s*\\|\', m.group(0))), text)\n\n    # 3. Cyrillic/Latin lookalikes → Cyrillic\n    text = \'\'.join(_LATIN_TO_CYRILLIC.get(ch, ch) for ch in text)\n\n    # 4. Dashes → hyphen-minus\n    text = _DASHES.sub(\'-\', text)\n\n    # 5. Filler dashes/underscores → 3 chars\n    text = _FILLERS.sub(\'___\', text)\n\n    # 6. Quotes\n    text = _QUOTES_DOUBLE.sub(\'"\', text)\n    text = _QUOTES_SINGLE.sub("\'", text)\n\n    # 7. Whitespace: NBSP and exotic spaces → regular space, collapse multiples\n    text = _MULTI_SPACE.sub(\' \', text)\n\n    # 8. Spaces inside parentheses\n    text = _SPACE_IN_PARENS.sub(\'(\', text)\n    text = _SPACE_IN_PARENS_R.sub(\')\', text)\n    # 8b. Strip trailing space introduced by `\\sin ` / `\\lim ` etc. when\n    # followed by "(" or sub/superscript marker.\n    text = _SPACE_BEFORE_PAREN.sub(\'(\', text)\n    text = _SPACE_BEFORE_SUBSUPER.sub(r\'\\1\', text)\n\n    # 9. Table-specific: PSV cleanup (LaTeX table already converted in step 2)\n    if region_type == "table":\n        text = _PSV_PIPE.sub(\'|\', text)\n        lines = text.split(\'\\n\')\n        lines = [line.strip(\'|\').strip() for line in lines]\n        text = \'\\n\'.join(line for line in lines if line)\n\n    # 10. Strip leading/trailing whitespace\n    text = text.strip()\n\n    return text\n\n\n# ── Levenshtein distance (pure Python, no external deps) ──────\n\ndef _levenshtein(s1: str, s2: str) -> int:\n    if len(s1) < len(s2):\n        return _levenshtein(s2, s1)\n    if len(s2) == 0:\n        return len(s1)\n    prev = list(range(len(s2) + 1))\n    for i, c1 in enumerate(s1):\n        curr = [i + 1]\n        for j, c2 in enumerate(s2):\n            curr.append(min(\n                prev[j + 1] + 1,\n                curr[j] + 1,\n                prev[j] + (c1 != c2),\n            ))\n        prev = curr\n    return prev[-1]\n\n\n# ── IoU ───────────────────────────────────────────────────────\n\ndef _compute_iou(bbox1, bbox2):\n    x1 = max(bbox1[0], bbox2[0])\n    y1 = max(bbox1[1], bbox2[1])\n    x2 = min(bbox1[2], bbox2[2])\n    y2 = min(bbox1[3], bbox2[3])\n    if x2 <= x1 or y2 <= y1:\n        return 0.0\n    intersection = (x2 - x1) * (y2 - y1)\n    area1 = max(0, bbox1[2] - bbox1[0]) * max(0, bbox1[3] - bbox1[1])\n    area2 = max(0, bbox2[2] - bbox2[0]) * max(0, bbox2[3] - bbox2[1])\n    union = area1 + area2 - intersection\n    return intersection / union if union > 0 else 0.0\n\n\n# ── Greedy IoU matching ───────────────────────────────────────\n\ndef _greedy_match(gt_regions, pred_regions, threshold=0.5):\n    pairs = []\n    for gi, g in enumerate(gt_regions):\n        for pi, p in enumerate(pred_regions):\n            iou = _compute_iou(g["bbox"], p["bbox"])\n            if iou >= threshold:\n                pairs.append((iou, gi, pi))\n    pairs.sort(key=lambda x: -x[0])\n\n    matched = []\n    used_gt, used_pred = set(), set()\n    for iou, gi, pi in pairs:\n        if gi not in used_gt and pi not in used_pred:\n            matched.append((gi, pi))\n            used_gt.add(gi)\n            used_pred.add(pi)\n\n    unmatched_gt = [i for i in range(len(gt_regions)) if i not in used_gt]\n    unmatched_pred = [i for i in range(len(pred_regions)) if i not in used_pred]\n    return matched, unmatched_gt, unmatched_pred\n\n\n# ── Scorable check ────────────────────────────────────────────\n\ndef _is_scorable(region):\n    if region.get("type", "handwritten") in ("image", "graph"):\n        return False\n    if region.get("language", "uk") == "other":\n        return False\n    if region.get("legibility", "legible") == "illegible":\n        return False\n    return True\n\n\n# ── Page text builder ─────────────────────────────────────────\n\ndef _build_page_text(regions, normalize=False, drop_indices=None):\n    """Build concatenated page text from regions.\n\n    `drop_indices` (optional): set of region indices to additionally exclude\n    beyond the standard _is_scorable filter. Used on the prediction side to\n    drop pred regions that match a non-scorable GT region (otherwise their\n    text inflates pred_page asymmetrically vs gt_page).\n    """\n    drop = drop_indices or set()\n    scorable = [\n        r for i, r in enumerate(regions)\n        if _is_scorable(r) and i not in drop\n    ]\n    # Bucketed reading order: cluster regions whose center_y is within ~15px\n    # (half a typical handwritten line height), then left-to-right by center_x.\n    # Stabilizes ordering when detection splits one GT line into multiple bboxes\n    # with slightly different top_y values, which would otherwise scramble the\n    # page-level text concatenation.\n    scorable.sort(key=lambda r: (\n        ((r["bbox"][1] + r["bbox"][3]) / 2) // 15,\n        (r["bbox"][0] + r["bbox"][2]) / 2,\n    ))\n    if normalize:\n        return "\\n".join(\n            _normalize_text(r.get("text", ""), r.get("type", "handwritten"))\n            for r in scorable\n        )\n    return "\\n".join(r.get("text", "") for r in scorable)\n\n\n# ── Parse regions JSON ────────────────────────────────────────\n\nVALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}\n\n\ndef _parse_regions(regions_str, image_name, is_submission=True):\n    label = "Submission" if is_submission else "Solution"\n    if pd.isna(regions_str) or regions_str == "":\n        return []\n    try:\n        regions = json.loads(regions_str)\n    except (json.JSONDecodeError, TypeError) as e:\n        raise ParticipantVisibleError(\n            f\'{label} for image "{image_name}": invalid JSON in regions column. Error: {e}\'\n        )\n    if not isinstance(regions, list):\n        raise ParticipantVisibleError(\n            f\'{label} for image "{image_name}": regions must be a JSON list, got {type(regions).__name__}\'\n        )\n    for i, r in enumerate(regions):\n        if not isinstance(r, dict):\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: must be a JSON object\'\n            )\n        if "bbox" not in r:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: missing "bbox" field\'\n            )\n        bbox = r["bbox"]\n        if not isinstance(bbox, list) or len(bbox) != 4:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: bbox must be [x1, y1, x2, y2]\'\n            )\n        rtype = r.get("type", "handwritten")\n        if is_submission and rtype not in VALID_TYPES:\n            raise ParticipantVisibleError(\n                f\'{label} for image "{image_name}", region {i}: invalid type "{rtype}". \'\n                f\'Must be one of: {", ".join(sorted(VALID_TYPES))}\'\n            )\n        # Defaults\n        r.setdefault("type", "handwritten")\n        r.setdefault("language", "uk")\n        r.setdefault("legibility", "legible")\n        r.setdefault("text", "")\n    return regions\n\n\n# ── Main scoring function ────────────────────────────────────\n\ndef score(\n    solution: pd.DataFrame,\n    submission: pd.DataFrame,\n    row_id_column_name: str,\n    w_det: float = 0.15,\n    w_cls: float = 0.05,\n    w_cer: float = 0.30,\n    w_page: float = 0.50,\n) -> float:\n    """\n    Ukrainian Handwritten Text Recognition (HTR) Competition Metric.\n\n    Evaluates end-to-end document understanding: region detection,\n    classification, and text transcription.\n\n    Score = 0.15 * Detection_F1 + 0.05 * ClassAcc + 0.30 * (1-CER) + 0.50 * (1-PageCER)\n\n    Components:\n      - Detection F1 (0.15): type-agnostic bbox matching at IoU >= 0.5\n      - Classification Accuracy (0.05): correct region type among IoU-matched pairs\n      - CER (0.30): per-region Character Error Rate on matched scorable regions\n      - Page CER (0.50): full-page text comparison, agnostic to bbox granularity\n\n    Score range: 0.0 to 1.0 (higher is better).\n\n    Submission: CSV with columns `image` and `regions`.\n    `regions` is a JSON list of detected regions per image:\n        [{"bbox": [x1,y1,x2,y2], "type": "handwritten", "text": "..."}]\n\n    Region types: handwritten, printed, formula, table, annotation, image, graph.\n    Use [] for images with no detections. All test images must be present.\n\n    Text normalization (applied to both GT and predictions before CER):\n      - Cyrillic/Latin lookalikes unified (Latin c → Cyrillic с)\n      - Dash types unified (em/en-dash → hyphen)\n      - Whitespace collapsed, quotes normalized\n      - Strikethrough markers removed: ~~old~~{new} → new\n      - Formula: x_{3} → x_3, x² → x^2\n      - Table: whitespace around pipes stripped\n\n    Regions excluded from CER (GT attributes, not required from participants):\n      type=image/graph, language=other, legibility=illegible\n\n    >>> import pandas as pd\n    >>> sol = pd.DataFrame({"image": ["a.jpg"], "regions": [\'[{"bbox":[0,0,100,50],"type":"handwritten","text":"hello"}]\']})\n    >>> sub = pd.DataFrame({"image": ["a.jpg"], "regions": [\'[{"bbox":[0,0,100,50],"type":"handwritten","text":"hello"}]\']})\n    >>> score(sol, sub, "image")\n    1.0\n    """\n    # Validate columns\n    if "regions" not in submission.columns:\n        raise ParticipantVisibleError(\n            \'Submission must have a "regions" column containing JSON-encoded region predictions.\'\n        )\n    if "regions" not in solution.columns:\n        raise ParticipantVisibleError(\'Solution is missing "regions" column.\')\n\n    # Check all solution images are in submission\n    sol_images = set(solution[row_id_column_name])\n    sub_images = set(submission[row_id_column_name])\n    missing = sol_images - sub_images\n    if missing:\n        examples = sorted(missing)[:5]\n        raise ParticipantVisibleError(\n            f\'Submission is missing {len(missing)} image(s). Examples: {examples}. \'\n            f\'Include all test images, even those with no predictions (use empty regions: []).\'\n        )\n\n    # Build lookup\n    sub_lookup = {}\n    for _, row in submission.iterrows():\n        img = row[row_id_column_name]\n        sub_lookup[img] = _parse_regions(row["regions"], img, is_submission=True)\n\n    # Accumulators\n    all_det_tp = 0\n    all_det_fp = 0\n    all_det_fn = 0\n    all_class_correct = 0\n    all_class_total = 0\n    all_cer_values = []\n    all_page_cers = []\n\n    for _, row in solution.iterrows():\n        img = row[row_id_column_name]\n        gt = _parse_regions(row["regions"], img, is_submission=False)\n        pred = sub_lookup.get(img, [])\n\n        # Match by IoU\n        matched, unmatched_gt, unmatched_pred = _greedy_match(gt, pred, threshold=0.5)\n\n        # Detection F1 (type-agnostic: measures bbox quality only)\n        all_det_tp += len(matched)\n        all_det_fp += len(unmatched_pred)\n        all_det_fn += len(unmatched_gt)\n\n        # Classification Accuracy\n        for gi, pi in matched:\n            all_class_total += 1\n            if gt[gi]["type"] == pred[pi]["type"]:\n                all_class_correct += 1\n\n        # CER (per-region, with text normalization)\n        for gi, pi in matched:\n            if _is_scorable(gt[gi]):\n                rtype = gt[gi].get("type", "handwritten")\n                gt_text = _normalize_text(gt[gi].get("text", ""), rtype)\n                pred_text = _normalize_text(pred[pi].get("text", ""), rtype)\n                cer_i = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)\n                all_cer_values.append(cer_i)\n\n        # Page CER (with text normalization).\n        # Drop pred regions matched to a non-scorable GT region — otherwise\n        # their text would inflate pred_page while GT side excludes them.\n        pred_drop = {pi for gi, pi in matched if not _is_scorable(gt[gi])}\n        gt_page = _build_page_text(gt, normalize=True)\n        pred_page = _build_page_text(pred, normalize=True, drop_indices=pred_drop)\n        if len(gt_page) > 0:\n            page_cer_i = _levenshtein(pred_page, gt_page) / len(gt_page)\n            all_page_cers.append(page_cer_i)\n\n    # Aggregate\n    det_prec = all_det_tp / max(all_det_tp + all_det_fp, 1)\n    det_rec = all_det_tp / max(all_det_tp + all_det_fn, 1)\n    det_f1 = 2 * det_prec * det_rec / max(det_prec + det_rec, 1e-9)\n\n    class_acc = all_class_correct / max(all_class_total, 1)\n\n    # Default CER = 1.0 (worst) when no regions matched — prevents free score for empty submissions\n    cer = sum(all_cer_values) / len(all_cer_values) if all_cer_values else 1.0\n    page_cer = sum(all_page_cers) / len(all_page_cers) if all_page_cers else 1.0\n\n    final_score = (\n        w_det * det_f1\n        + w_cls * class_acc\n        + w_cer * max(0.0, 1.0 - cer)\n        + w_page * max(0.0, 1.0 - page_cer)\n    )\n\n    return float(final_score)\n\ndef score_detailed(\n    solution: pd.DataFrame,\n    submission: pd.DataFrame,\n    row_id_column_name: str,\n    w_det: float = 0.15,\n    w_cls: float = 0.05,\n    w_cer: float = 0.30,\n    w_page: float = 0.50,\n) -> dict:\n    """\n    Same metric as `score()` but returns a dict with all component scores.\n\n    Use this locally to debug your submission — you will see which component\n    (detection, classification, per-region CER, or page CER) is dragging\n    the score down.\n\n    Not used by Kaggle — Kaggle calls `score()` which returns a single float.\n    """\n    if "regions" not in submission.columns:\n        raise ParticipantVisibleError(\n            \'Submission must have a "regions" column containing JSON-encoded region predictions.\'\n        )\n    if "regions" not in solution.columns:\n        raise ParticipantVisibleError(\'Solution is missing "regions" column.\')\n\n    sol_images = set(solution[row_id_column_name])\n    sub_images = set(submission[row_id_column_name])\n    missing = sol_images - sub_images\n    if missing:\n        examples = sorted(missing)[:5]\n        raise ParticipantVisibleError(\n            f\'Submission is missing {len(missing)} image(s). Examples: {examples}.\'\n        )\n\n    sub_lookup = {\n        row[row_id_column_name]: _parse_regions(row["regions"], row[row_id_column_name], is_submission=True)\n        for _, row in submission.iterrows()\n    }\n\n    all_det_tp = all_det_fp = all_det_fn = 0\n    all_class_correct = all_class_total = 0\n    all_cer_values, all_page_cers = [], []\n\n    for _, row in solution.iterrows():\n        img = row[row_id_column_name]\n        gt = _parse_regions(row["regions"], img, is_submission=False)\n        pred = sub_lookup.get(img, [])\n        matched, unmatched_gt, unmatched_pred = _greedy_match(gt, pred, threshold=0.5)\n\n        all_det_tp += len(matched)\n        all_det_fp += len(unmatched_pred)\n        all_det_fn += len(unmatched_gt)\n\n        for gi, pi in matched:\n            all_class_total += 1\n            if gt[gi]["type"] == pred[pi]["type"]:\n                all_class_correct += 1\n\n        for gi, pi in matched:\n            if _is_scorable(gt[gi]):\n                rtype = gt[gi].get("type", "handwritten")\n                gt_text = _normalize_text(gt[gi].get("text", ""), rtype)\n                pred_text = _normalize_text(pred[pi].get("text", ""), rtype)\n                cer_i = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)\n                all_cer_values.append(cer_i)\n\n        # Drop pred regions matched to non-scorable GT (avoid asymmetric pred_page inflation)\n        pred_drop = {pi for gi, pi in matched if not _is_scorable(gt[gi])}\n        gt_page = _build_page_text(gt, normalize=True)\n        pred_page = _build_page_text(pred, normalize=True, drop_indices=pred_drop)\n        if len(gt_page) > 0:\n            all_page_cers.append(_levenshtein(pred_page, gt_page) / len(gt_page))\n\n    det_prec = all_det_tp / max(all_det_tp + all_det_fp, 1)\n    det_rec = all_det_tp / max(all_det_tp + all_det_fn, 1)\n    det_f1 = 2 * det_prec * det_rec / max(det_prec + det_rec, 1e-9)\n    class_acc = all_class_correct / max(all_class_total, 1)\n    cer = sum(all_cer_values) / len(all_cer_values) if all_cer_values else 1.0\n    page_cer = sum(all_page_cers) / len(all_page_cers) if all_page_cers else 1.0\n\n    composite = (\n        w_det * det_f1\n        + w_cls * class_acc\n        + w_cer * max(0.0, 1.0 - cer)\n        + w_page * max(0.0, 1.0 - page_cer)\n    )\n\n    return {\n        "composite_score": float(composite),\n        "detection_f1": float(det_f1),\n        "detection_precision": float(det_prec),\n        "detection_recall": float(det_rec),\n        "classification_accuracy": float(class_acc),\n        "region_cer": float(cer),\n        "page_cer": float(page_cer),\n        "n_images": int(len(solution)),\n        "n_matched_regions": int(all_det_tp),\n        "n_false_positives": int(all_det_fp),\n        "n_false_negatives": int(all_det_fn),\n    }\n\n\n# ── CLI for local debugging ───────────────────────────────────\n# NOTE: kept as a function (not a top-level __main__ block) so that\n# importing this file in a notebook / Kaggle metric uploader does not\n# trigger argparse on a kernel-launcher cmdline.\n\ndef _cli_main():\n    import argparse\n    parser = argparse.ArgumentParser(\n        description="RUKOPYS scoring — run locally to see component breakdown",\n    )\n    parser.add_argument("--solution", required=True, help="Path to ground-truth CSV (image, regions)")\n    parser.add_argument("--submission", required=True, help="Path to your submission CSV (image, regions)")\n    parser.add_argument("--row-id", default="image", help="Row ID column name (default: image)")\n    args = parser.parse_args()\n\n    sol = pd.read_csv(args.solution)\n    sub = pd.read_csv(args.submission)\n\n    r = score_detailed(sol, sub, args.row_id)\n\n    print()\n    print(f"  Images evaluated       : {r[\'n_images\']}")\n    print(f"  Matched regions (IoU≥.5): {r[\'n_matched_regions\']}")\n    print(f"  False positives        : {r[\'n_false_positives\']}")\n    print(f"  False negatives        : {r[\'n_false_negatives\']}")\n    print()\n    print(f"  Detection F1           : {r[\'detection_f1\']:.4f}   (precision {r[\'detection_precision\']:.3f} / recall {r[\'detection_recall\']:.3f})")\n    print(f"  Classification accuracy: {r[\'classification_accuracy\']:.4f}")\n    print(f"  Region CER             : {r[\'region_cer\']:.4f}   → score {1-r[\'region_cer\']:.4f}")\n    print(f"  Page CER               : {r[\'page_cer\']:.4f}   → score {1-r[\'page_cer\']:.4f}")\n    print(f"  ──────────────────────────────────────────────────")\n    print(f"  Composite score        : {r[\'composite_score\']:.4f}")\n    print()\n\n\nif __name__ == "__main__":\n    import sys\n    # Only auto-run CLI if --solution arg present; skips Jupyter/Colab\n    # where __name__ == "__main__" but sys.argv is a kernel launcher.\n    if any(a == "--solution" for a in sys.argv[1:]):\n        _cli_main()\n'
metric_path = Path('kaggle_metric.py').resolve()
metric_path.write_text(KAGGLE_METRIC_SOURCE, encoding='utf-8')
if str(metric_path.parent) not in sys.path:
    sys.path.insert(0, str(metric_path.parent))
print(f'Wrote {metric_path} from official notebook source')


Wrote C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\kaggle_metric.py from official notebook source


In [18]:
import csv
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from kaggle_metric import (
    ParticipantVisibleError,
    _build_page_text,
    _greedy_match,
    _is_scorable,
    _levenshtein,
    _normalize_text,
    _parse_regions,
    score_detailed,
)

VALID_TYPES = ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
SCORABLE_TYPES = ["handwritten", "printed", "formula", "table", "annotation"]
QWEN_TYPES = {"formula", "table"}

RUN_METADATA = {
    "00_yolo_bbox_only_empty_text": ("full_system", "YOLO bbox only, empty text"),
    "01_yolo_bbox_type_empty_text": ("full_system", "YOLO bbox + type, empty text"),
    "02_yolo_trocr_all_text_like": ("full_system", "YOLO + TrOCR for all text-like regions"),
    "03_yolo_qwen_all_text_like": ("full_system", "YOLO + Qwen3-VL for all text-like regions"),
    "04_yolo_trocr_hpa_empty_formula_table": ("full_system", "YOLO + TrOCR HPA, empty formula/table"),
    "05_yolo_qwen_formula_table_empty_hpa": ("full_system", "YOLO + Qwen formula/table, empty HPA"),
    "06_yolo_trocr_qwen_hybrid": ("full_system", "Ours: YOLO + TrOCR + Qwen routing"),
    "01_yolo_trocr_all_text_like": ("routing_ablation", "TrOCR for all text-like regions"),
    "02_yolo_qwen_all_text_like": ("routing_ablation", "Qwen3-VL for all text-like regions"),
    "03_yolo_trocr_hpa_empty_formula_table": ("routing_ablation", "TrOCR for HPA, empty formula/table"),
    "04_yolo_qwen_formula_table_empty_hpa": ("routing_ablation", "Qwen for formula/table, empty HPA"),
    "05_yolo_trocr_qwen_hybrid": ("routing_ablation", "Hybrid routing: TrOCR HPA + Qwen formula/table"),
    "01_base_qwen_generic_gt_formula_table": ("qwen_ablation", "Base Qwen3-VL, zero-shot generic prompt"),
    "02_lora_generic_gt_formula_table": ("qwen_ablation", "LoRA, generic prompt"),
    "03_lora_type_specific_gt_formula_table": ("qwen_ablation", "LoRA, type-specific prompt"),
    "04_lora_source_type_gt_formula_table": ("qwen_ablation", "LoRA, source-aware + type-specific prompt"),
    "05_lora_source_guardrails_gt_formula_table": ("qwen_ablation", "LoRA, source-aware prompt + guardrails"),
    "06_lora_final_gt_formula_table": ("qwen_ablation", "LoRA, source-aware + type-specific prompt + guardrails"),
    "01_detector_yolo_predictions_empty_text": ("module_results", "DocLayout-YOLO detector predictions"),
    "02_hpa_gt_crops": ("module_results", "TrOCR HPA on GT crops"),
    "03_qwen_final_gt_formula_table": ("module_results", "Qwen3-VL final prompt on GT formula/table"),
}

FULL_SYSTEM_ORDER = [
    "YOLO bbox only, empty text",
    "YOLO bbox + type, empty text",
    "YOLO + TrOCR for all text-like regions",
    "YOLO + Qwen3-VL for all text-like regions",
    "YOLO + TrOCR HPA, empty formula/table",
    "YOLO + Qwen formula/table, empty HPA",
    "Ours: YOLO + TrOCR + Qwen routing",
]
ROUTING_ORDER = [
    "TrOCR for all text-like regions",
    "Qwen3-VL for all text-like regions",
    "TrOCR for HPA, empty formula/table",
    "Qwen for formula/table, empty HPA",
    "Hybrid routing: TrOCR HPA + Qwen formula/table",
]
QWEN_ORDER = [
    "Base Qwen3-VL, zero-shot generic prompt",
    "LoRA, generic prompt",
    "LoRA, type-specific prompt",
    "LoRA, source-aware + type-specific prompt",
    "LoRA, source-aware prompt + guardrails",
    "LoRA, source-aware + type-specific prompt + guardrails",
]


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def clean_region_for_solution(region):
    cleaned = {
        "bbox": region.get("bbox", [0, 0, 0, 0]),
        "type": str(region.get("type") or "handwritten"),
        "text": str(region.get("text") or ""),
    }
    for key in ["language", "legibility"]:
        if key in region:
            cleaned[key] = region[key]
    return cleaned


def build_solution_df(gt_jsonl):
    records = read_jsonl(gt_jsonl)
    rows = []
    for record in records:
        image = Path(str(record.get("file_name") or record.get("image") or "")).name
        regions = [clean_region_for_solution(r) for r in record.get("regions", [])]
        rows.append({"image": image, "regions": json.dumps(regions, ensure_ascii=False)})
    return pd.DataFrame(rows), records


def infer_run_id(path):
    path = Path(path)
    stem = path.stem
    if stem.endswith("_submission"):
        return stem[: -len("_submission")]
    if path.name == "submission.csv":
        return path.parent.name
    return stem


def discover_submission_csvs(search_dirs, extra_files):
    paths = []
    for item in search_dirs:
        root = Path(item)
        if not root.exists():
            continue
        if root.is_file() and root.suffix.lower() == ".csv":
            paths.append(root)
            continue
        for p in root.rglob("*.csv"):
            lower = p.name.lower()
            if "submission" not in lower:
                continue
            if any(x in lower for x in ["summary", "metrics", "per_type", "per_source", "details"]):
                continue
            paths.append(p)
    for item in extra_files:
        p = Path(item)
        if p.exists():
            paths.append(p)
    unique = []
    seen = set()
    for p in paths:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return sorted(unique, key=lambda p: str(p))


def safe_read_submission(path):
    df = pd.read_csv(path)
    if "image" not in df.columns or "regions" not in df.columns:
        raise ValueError(f"{path} phải có cột image và regions")
    df = df[["image", "regions"]].copy()
    df["image"] = df["image"].astype(str)
    df["regions"] = df["regions"].fillna("[]").astype(str)
    return df


def ordered(df, label_col, order):
    if df.empty:
        return df
    rank = {label: i for i, label in enumerate(order)}
    return df.assign(_rank=df[label_col].map(lambda x: rank.get(x, 999))).sort_values(["_rank", label_col]).drop(columns=["_rank"])


def fmt_float(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.{ROUND_DIGITS}f}"


def markdown_cell(value):
    if pd.isna(value):
        text = ""
    else:
        text = str(value)
    return text.replace("|", "\\|").replace("\n", "<br>")


def save_markdown_table(df, path):
    path = Path(path)
    if df.empty:
        path.write_text("_No rows._\n", encoding="utf-8")
        return
    columns = list(df.columns)
    header = "| " + " | ".join(markdown_cell(c) for c in columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for _, row in df.iterrows():
        body.append("| " + " | ".join(markdown_cell(row[c]) for c in columns) + " |")
    path.write_text("\n".join([header, sep, *body]) + "\n", encoding="utf-8")

def invalid_reason(text, rtype, gt_text=""):
    raw = str(text or "").strip()
    lower = raw.lower()
    if not raw:
        return "empty"
    if raw.startswith("```") or "```" in raw:
        return "markdown_fence"
    if raw.startswith("{") or raw.startswith("["):
        return "json_or_list_wrapper"
    if re.match(r"(?i)^\s*(text|transcription|answer|output)\s*:", raw):
        return "prefixed_label"
    if re.search(r"(?i)\b(i can see|the image|this is|the answer|solution|solve|simplif)", raw):
        return "explanation_or_solution_like"
    if rtype == "table":
        gt_has_table_sep = "|" in str(gt_text) or "\n" in str(gt_text)
        pred_has_table_sep = "|" in raw or "\n" in raw
        if gt_has_table_sep and not pred_has_table_sep:
            return "missing_table_separator"
        lines = [line for line in raw.splitlines() if line.strip()]
        if len(lines) > 1 and any("|" not in line for line in lines):
            return "malformed_table_rows"
    return ""


def collect_breakdowns(solution_df, submission_df, run_id, group, label):
    sub_lookup = {
        row[ROW_ID_COLUMN]: _parse_regions(row["regions"], row[ROW_ID_COLUMN], is_submission=True)
        for _, row in submission_df.iterrows()
    }
    gt_lookup = {}
    source_by_image = {}
    for record in GT_RECORDS:
        image = Path(str(record.get("file_name") or record.get("image") or "")).name
        source_by_image[image] = str(record.get("source") or "unknown")

    per_type = defaultdict(lambda: Counter())
    per_source = defaultdict(lambda: Counter())
    cer_by_type = defaultdict(list)
    cer_by_source = defaultdict(list)
    qwen_rows = []

    for _, row in solution_df.iterrows():
        image = row[ROW_ID_COLUMN]
        source = source_by_image.get(image, "unknown")
        gt = _parse_regions(row["regions"], image, is_submission=False)
        pred = sub_lookup.get(image, [])
        matched, unmatched_gt, unmatched_pred = _greedy_match(gt, pred, threshold=IOU_THRESHOLD)

        for g in gt:
            per_type[g.get("type", "handwritten")]["gt_count"] += 1
            per_source[source]["gt_count"] += 1
        for p in pred:
            per_type[p.get("type", "handwritten")]["pred_count"] += 1
            per_source[source]["pred_count"] += 1

        for gi, pi in matched:
            gt_type = gt[gi].get("type", "handwritten")
            pred_type = pred[pi].get("type", "handwritten")
            per_type[gt_type]["matched"] += 1
            per_source[source]["matched"] += 1
            if gt_type == pred_type:
                per_type[gt_type]["type_correct"] += 1
                per_source[source]["type_correct"] += 1
            if _is_scorable(gt[gi]):
                gt_text_raw = gt[gi].get("text", "")
                pred_text_raw = pred[pi].get("text", "")
                gt_text = _normalize_text(gt_text_raw, gt_type)
                pred_text = _normalize_text(pred_text_raw, gt_type)
                cer_i = _levenshtein(pred_text, gt_text) / max(len(gt_text), 1)
                cer_by_type[gt_type].append(cer_i)
                cer_by_source[source].append(cer_i)
                if gt_type in QWEN_TYPES:
                    reason = invalid_reason(pred_text_raw, gt_type, gt_text_raw)
                    qwen_rows.append({
                        "run_id": run_id,
                        "group": group,
                        "label": label,
                        "image": image,
                        "type": gt_type,
                        "cer": cer_i,
                        "invalid": bool(reason),
                        "invalid_reason": reason,
                        "gt_text": gt_text_raw,
                        "pred_text": pred_text_raw,
                    })

        for gi in unmatched_gt:
            gt_type = gt[gi].get("type", "handwritten")
            per_type[gt_type]["fn"] += 1
            per_source[source]["fn"] += 1
        for pi in unmatched_pred:
            pred_type = pred[pi].get("type", "handwritten")
            per_type[pred_type]["fp"] += 1
            per_source[source]["fp"] += 1

    per_type_rows = []
    for rtype in VALID_TYPES:
        c = per_type[rtype]
        tp = c["matched"]
        fp = c["fp"]
        fn = c["fn"]
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        per_type_rows.append({
            "run_id": run_id,
            "group": group,
            "label": label,
            "type": rtype,
            "gt_count": int(c["gt_count"]),
            "pred_count": int(c["pred_count"]),
            "matched": int(tp),
            "det_f1": f1,
            "class_acc": c["type_correct"] / max(tp, 1),
            "cer": sum(cer_by_type[rtype]) / len(cer_by_type[rtype]) if cer_by_type[rtype] else math.nan,
        })

    per_source_rows = []
    for source, c in sorted(per_source.items()):
        tp = c["matched"]
        fp = c["fp"]
        fn = c["fn"]
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        per_source_rows.append({
            "run_id": run_id,
            "group": group,
            "label": label,
            "source": source,
            "gt_count": int(c["gt_count"]),
            "pred_count": int(c["pred_count"]),
            "det_f1": f1,
            "class_acc": c["type_correct"] / max(tp, 1),
            "cer": sum(cer_by_source[source]) / len(cer_by_source[source]) if cer_by_source[source] else math.nan,
        })

    qwen_df = pd.DataFrame(qwen_rows)
    qwen_summary = {}
    if not qwen_df.empty:
        for rtype in ["formula", "table"]:
            part = qwen_df[qwen_df["type"] == rtype]
            qwen_summary[f"{rtype}_cer"] = float(part["cer"].mean()) if not part.empty else math.nan
        qwen_summary["struct_acc"] = float((~qwen_df["invalid"]).mean())
        qwen_summary["invalid_rate"] = float(qwen_df["invalid"].mean())
        qwen_summary["qwen_matched_count"] = int(len(qwen_df))
    else:
        qwen_summary = {
            "formula_cer": math.nan,
            "table_cer": math.nan,
            "struct_acc": math.nan,
            "invalid_rate": math.nan,
            "qwen_matched_count": 0,
        }

    return pd.DataFrame(per_type_rows), pd.DataFrame(per_source_rows), qwen_df, qwen_summary


In [19]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
solution_df, GT_RECORDS = build_solution_df(GT_JSONL)
solution_csv = OUTPUT_DIR / "solution_from_test_jsonl.csv"
solution_df.to_csv(solution_csv, index=False)
print(f"Ghi solution CSV: {solution_csv} rows={len(solution_df)}")

submission_paths = discover_submission_csvs(SUBMISSION_SEARCH_DIRS, EXTRA_SUBMISSION_FILES)
print(f"Tìm thấy {len(submission_paths)} submission CSV")
for p in submission_paths:
    print("-", p)

all_rows = []
all_per_type = []
all_per_source = []
all_qwen_details = []
errors = []

for path in submission_paths:
    run_id = infer_run_id(path)
    group, label = RUN_METADATA.get(run_id, ("unknown", run_id))
    print(f"\nĐang chấm: {run_id} [{group}] -> {path}")
    try:
        sub_df = safe_read_submission(path)
        detail = score_detailed(solution_df, sub_df, ROW_ID_COLUMN)
        per_type_df, per_source_df, qwen_detail_df, qwen_summary = collect_breakdowns(solution_df, sub_df, run_id, group, label)
        row = {
            "run_id": run_id,
            "group": group,
            "label": label,
            "path": str(path),
            "official_score": detail["composite_score"],
            "det_f1": detail["detection_f1"],
            "det_precision": detail["detection_precision"],
            "det_recall": detail["detection_recall"],
            "class_acc": detail["classification_accuracy"],
            "region_cer": detail["region_cer"],
            "page_cer": detail["page_cer"],
            "matched_regions": detail["n_matched_regions"],
            "false_positives": detail["n_false_positives"],
            "false_negatives": detail["n_false_negatives"],
            **qwen_summary,
        }
        all_rows.append(row)
        all_per_type.append(per_type_df)
        all_per_source.append(per_source_df)
        if not qwen_detail_df.empty:
            all_qwen_details.append(qwen_detail_df)
        print(f"  score={detail['composite_score']:.4f} det_f1={detail['detection_f1']:.4f} class_acc={detail['classification_accuracy']:.4f} region_cer={detail['region_cer']:.4f} page_cer={detail['page_cer']:.4f}")
    except Exception as exc:
        print(f"  LỖI: {exc}")
        errors.append({"run_id": run_id, "path": str(path), "error": repr(exc)})

summary_df = pd.DataFrame(all_rows)
per_type_df = pd.concat(all_per_type, ignore_index=True) if all_per_type else pd.DataFrame()
per_source_df = pd.concat(all_per_source, ignore_index=True) if all_per_source else pd.DataFrame()
qwen_details_df = pd.concat(all_qwen_details, ignore_index=True) if all_qwen_details else pd.DataFrame()
errors_df = pd.DataFrame(errors)

summary_path = OUTPUT_DIR / "all_submission_metrics.csv"
per_type_path = OUTPUT_DIR / "per_type_metrics.csv"
per_source_path = OUTPUT_DIR / "per_source_metrics.csv"
qwen_details_path = OUTPUT_DIR / "qwen_invalid_and_cer_details.csv"
errors_path = OUTPUT_DIR / "evaluation_errors.csv"

summary_df.to_csv(summary_path, index=False)
per_type_df.to_csv(per_type_path, index=False)
per_source_df.to_csv(per_source_path, index=False)
qwen_details_df.to_csv(qwen_details_path, index=False)
errors_df.to_csv(errors_path, index=False)

print("\nĐã ghi:")
for p in [summary_path, per_type_path, per_source_path, qwen_details_path, errors_path]:
    print("-", p)


Ghi solution CSV: C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\solution_from_test_jsonl.csv rows=200
Tìm thấy 7 submission CSV
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\module_results\03_qwen_final_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\01_base_qwen_generic_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\02_lora_generic_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\03_lora_type_specific_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\04_lora_source_type_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\05_lora_source_guardrails_gt_formula_table_submission.csv
- C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\qwen_ablation\06_lora_final_gt

In [20]:
def metric_cols_for_paper(df):
    cols = ["label", "det_f1", "class_acc", "region_cer", "page_cer", "official_score"]
    out = df[cols].copy()
    for c in cols[1:]:
        out[c] = out[c].map(fmt_float)
    return out

def build_module_results_table(qwen_details_df):
    rows = []
    if qwen_details_df.empty:
        return pd.DataFrame(columns=["Component", "Split / Group", "Metric", "N", "Value"])
    qwen_module = qwen_details_df[qwen_details_df["run_id"] == "03_qwen_final_gt_formula_table"].copy()
    for rtype in ["formula", "table"]:
        part = qwen_module[qwen_module["type"] == rtype]
        rows.append({
            "Component": "Qwen3-VL",
            "Split / Group": rtype,
            "Metric": "CER",
            "N": int(len(part)),
            "Value": fmt_float(part["cer"].mean()) if not part.empty else "",
        })
    return pd.DataFrame(rows)

if summary_df.empty:
    print("Chưa có submission nào để tạo bảng.")
else:
    full_system_df = ordered(summary_df[summary_df["group"] == "full_system"], "label", FULL_SYSTEM_ORDER)
    routing_df = ordered(summary_df[summary_df["group"] == "routing_ablation"], "label", ROUTING_ORDER)
    qwen_df = ordered(summary_df[summary_df["group"] == "qwen_ablation"], "label", QWEN_ORDER)
    module_df = summary_df[summary_df["group"] == "module_results"].copy()

    full_system_table = metric_cols_for_paper(full_system_df) if not full_system_df.empty else pd.DataFrame()
    routing_table = metric_cols_for_paper(routing_df) if not routing_df.empty else pd.DataFrame()

    if not qwen_df.empty:
        qwen_table = qwen_df[["label", "official_score", "formula_cer", "table_cer", "struct_acc", "invalid_rate", "qwen_matched_count"]].copy()
        for c in ["official_score", "formula_cer", "table_cer", "struct_acc", "invalid_rate"]:
            qwen_table[c] = qwen_table[c].map(fmt_float)
        qwen_debug_table = qwen_df[["label", "official_score", "region_cer", "page_cer", "formula_cer", "table_cer", "qwen_matched_count"]].copy()
        for c in ["official_score", "region_cer", "page_cer", "formula_cer", "table_cer"]:
            qwen_debug_table[c] = qwen_debug_table[c].map(fmt_float)
    else:
        qwen_table = pd.DataFrame()
        qwen_debug_table = pd.DataFrame()

    module_table = build_module_results_table(qwen_details_df)

    table_paths = {
        "paper_full_system_results.csv": full_system_table,
        "paper_routing_ablation.csv": routing_table,
        "paper_qwen_ablation.csv": qwen_table,
        "paper_qwen_ablation_full_official_debug.csv": qwen_debug_table,
        "paper_module_results.csv": module_table,
    }
    for filename, df in table_paths.items():
        csv_path = OUTPUT_DIR / filename
        md_path = OUTPUT_DIR / filename.replace(".csv", ".md")
        df.to_csv(csv_path, index=False)
        save_markdown_table(df, md_path)
        print(f"Ghi {csv_path}")
        print(f"Ghi {md_path}")

    print("\n=== Full-system table ===")
    display(full_system_table)
    print("\n=== Routing ablation table ===")
    display(routing_table)
    print("\n=== Qwen ablation table ===")
    display(qwen_table)
    print("\n=== Module table ===")
    display(module_table)

    if not per_type_df.empty:
        final_like = summary_df[summary_df["label"].str.contains("Ours:|Hybrid routing|Qwen3-VL final", regex=True, na=False)][["run_id", "label"]]
        print("\nGợi ý: dùng per_type_metrics.csv để điền per-type results. Các run final/hybrid hiện có:")
        display(final_like)

    if not qwen_details_df.empty:
        invalid_examples = qwen_details_df[qwen_details_df["invalid"]].head(50)
        invalid_examples_path = OUTPUT_DIR / "qwen_invalid_examples_top50.csv"
        invalid_examples.to_csv(invalid_examples_path, index=False)
        print(f"\nGhi ví dụ invalid Qwen: {invalid_examples_path}")
        display(invalid_examples[["run_id", "image", "type", "invalid_reason", "gt_text", "pred_text"]].head(10))


Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_full_system_results.csv
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_full_system_results.md
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_routing_ablation.csv
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_routing_ablation.md
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_qwen_ablation.csv
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_qwen_ablation.md
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_qwen_ablation_full_official_debug.csv
Ghi C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\paper_qwen_ablation_full_official_debug.md
Ghi C:\Data\Workspace\Handwritten_

""



=== Routing ablation table ===


""



=== Qwen ablation table ===


...

,Component,Split / Group,Metric,N,Value
0,Qwen3-VL,formula,CER,449,0.2019
1,Qwen3-VL,table,CER,12,0.2359



Gợi ý: dùng per_type_metrics.csv để điền per-type results. Các run final/hybrid hiện có:


,run_id,label
0,03_qwen_final_gt_formula_table,Qwen3-VL final prompt on GT formula/table



Ghi ví dụ invalid Qwen: C:\Data\Workspace\Handwritten_to_Data_Ukraine\fillpaper\evaluate\paper_metric_outputs\qwen_invalid_examples_top50.csv


,run_id,image,type,invalid_reason,gt_text,pred_text
23,03_qwen_final_gt_formula_table,26e0c337-b806-4a49-b9c2-456fb67708aa.jpg,formula,empty,\lfloor \frac{2020}{7} \rfloor = 288,
24,03_qwen_final_gt_formula_table,26e0c337-b806-4a49-b9c2-456fb67708aa.jpg,formula,empty,\left[ \frac{2020}{29} \right] = 69,
26,03_qwen_final_gt_formula_table,26e0c337-b806-4a49-b9c2-456fb67708aa.jpg,formula,empty,\left[ \frac{2020}{203} \right] = 9,
72,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"{0}={6, 12, 18~~}~~}",
73,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"{1}={1, 7, 13}",
74,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"[2} = {2, 8, 14}",
75,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"[3] = {3, 9, 15}",
76,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"[4] ={4,10,16}",
77,03_qwen_final_gt_formula_table,2a35bd24-d0cd-4974-89ac-42e21de368ec.jpg,formula,empty,"[5] = {5, 11, 17}",
89,03_qwen_final_gt_formula_table,40f7d696-b0de-4c43-a8fc-fff4c426cc83.jpg,formula,empty,{y > 2x + 1,
